# Week 7 — SMOTE Pipeline Demo
Παράδειγμα χωρίς leakage: scaling + SMOTE μέσα σε Pipeline με StratifiedKFold.

Αν χρειαστεί κάνε install το 
 - python -m pip install -U imbalanced-learn
(αν θες, και scikit-learn ενημέρωση:)
- (.venv) ...> python -m pip install -U scikit-learn

### Φορτώνουμε:
- LogisticRegression: baseline γραμμικό μοντέλο με πιθανότητες.
- SMOTE: συνθετικό oversampling της μειοψηφίας.
- Pipeline (imblearn): εκτελεί σωστά fit_resample (SMOTE) μέσα στο κάθε CV fold → καμία διαρροή.
- StandardScaler: κλιμάκωση χαρακτηριστικών πριν το SMOTE (ώστε οι αποστάσεις για τους γείτονες να είναι συγκρίσιμες) και πριν το μοντέλο.
- StratifiedKFold / cross_val_score: στρωματοποιημένη CV για σπάνια κλάση.
- average_precision_score: PR-AUC (Average Precision) — κατάλληλο για rare positives.

In [6]:
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import make_scorer, average_precision_score
# Ορίζουμε partial oversampling: δεν εξισώνουμε πλήρως τις κλάσεις (που θα «φούσκωνε» πολύ το train και θα ρίσκαρε overfitting). 5% σημαίνει λόγος περίπου 1:20 – συντηρητική, πιο business-friendly επιλογή.
smote = SMOTE(
    random_state=42,
    sampling_strategy=0.05  # minority = 5% του πλήθους της majority
)
""" Scaling → σωστές αποστάσεις για SMOTE & σταθερότερη βελτιστοποίηση της LogReg.
    SMOTE → δημιουργεί συνθετικά μόνο στο training fold.
    LogReg(class_weight="balanced") → δίνει επιπλέον βάρος στη μειοψηφία ώστε να μειώσουμε FN χωρίς να γεμίσουμε από FP."""
pipe = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("smote", smote),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42) # Κάθε fold κρατά το ποσοστό απάτης ίδιο με το συνολικό.
ap_scorer = make_scorer(average_precision_score, needs_proba=True) # PR-AUC
 
scores = cross_val_score(
    pipe, X, y,
    scoring="average_precision",   # built-in PR-AUC
    cv=cv,
    n_jobs=-1,
    error_score="raise"
)
print(f"PR-AUC (CV mean ± std): {scores.mean():.4f} ± {scores.std():.4f}")


PR-AUC (CV mean ± std): 0.7426 ± 0.0210


- Partial SMOTE + balanced weights → καλή ισορροπία ανάμεσα σε Recall και διαχειρίσιμο όγκο FP για λειτουργία.
- PR-AUC είναι η βασική συνολική μετρική για rare positives. Παράλληλα κοίτα Recall@k (π.χ. top-N alerts που μπορεί να χειριστεί η ομάδα).
- Αν το training αργεί/βαραίνει, ρίξε sampling_strategy σε 0.02 (1:50) και σύγκρινε PR-AUC/Recall.

PR-AUC (CV mean ± std): 0.7426 ± 0.0210
### Τι σημαίνει αυτό
- Τυχαίο baseline (prevalence) ≈ ποσοστό απάτης ≈ 0.00167.
Άρα το 0.7426 είναι πολλές τάξεις μεγέθους καλύτερο από το τυχαίο—σούπερ δυνατό σήμα για rare positives.
- Std = 0.021 → η απόδοση είναι σταθερή μεταξύ folds (δεν “κρέμεται” από ένα fold).